In [0]:
%sql
WITH src AS (
    SELECT * FROM pep_dev.bronze.statements
    WHERE _snapshot_version = (SELECT MAX(_snapshot_version) FROM pep_dev.bronze.statements)
      AND schema = 'Occupancy'
      AND prop IN ('holder', 'post', 'startDate', 'status', 'endDate')
),

agg AS (
    SELECT
        canonical_id AS occupancy_id,
        max(value) FILTER (WHERE prop = 'holder') AS person_ref,
        max(value) FILTER (WHERE prop = 'post') AS post_ref,
        min(value) FILTER (WHERE prop = 'startDate') AS start_raw,
        coalesce(
            CASE WHEN length(start_raw) = 4 THEN 'year' END,
            CASE WHEN length(start_raw) = 7 THEN 'month' END,
            CASE WHEN length(start_raw) = 10 THEN 'day' END,
            NULL
        ) AS start_precision,
        max(value) FILTER (WHERE prop = 'endDate') AS end_raw,
        coalesce(
            CASE WHEN length(end_raw) = 4 THEN 'year' END,
            CASE WHEN length(end_raw) = 7 THEN 'month' END,
            CASE WHEN length(end_raw) = 10 THEN 'day' END,
            NULL
        ) AS end_precision,
        try_cast(start_raw AS DATE) AS start_date,
        try_cast(end_raw AS DATE) AS end_date,
        max(value) FILTER (WHERE prop = 'status') AS status_raw,
        collect_set(dataset) AS datasets,
        size(datasets) AS source_count,
        max(_snapshot_version) AS _snapshot_version,
        to_date(left(max(_snapshot_version), 8), 'yyyyMMdd') AS snapshot_date,
        CASE end_precision
            WHEN 'year'  THEN make_date(year(end_date), 12, 31)
            WHEN 'month' THEN last_day(end_date)
            ELSE end_date
        END AS end_date_upper,
        CASE
            WHEN end_date_upper < snapshot_date THEN 'ended'
            WHEN lower(trim(status_raw)) IN ('current', 'ended', 'unknown') THEN lower(trim(status_raw))
            ELSE 'unknown'
        END AS status_norm,
        coalesce(end_date_upper < start_date, false) AS date_conflict
    FROM src
    GROUP BY canonical_id
    HAVING count_if(prop = 'holder') > 0
),

final AS (
    SELECT
        o.occupancy_id,
        o.person_ref,
        CASE WHEN EXISTS (SELECT 1 FROM pep_dev.silver.person p WHERE p.person_id = o.person_ref)
             THEN o.person_ref END AS person_id,
        o.post_ref,
        CASE WHEN EXISTS (SELECT 1 FROM pep_dev.silver.post pt WHERE pt.post_id = o.post_ref)
             THEN o.post_ref END AS post_id,
        o.start_raw,
        o.start_precision,
        o.start_date,
        o.end_raw,
        o.end_precision,
        o.end_date,
        o.status_raw,
        o.status_norm,
        o.date_conflict,
        o.datasets,
        o.source_count,
        o._snapshot_version
    FROM agg o
)

MERGE INTO pep_dev.silver.occupancy AS t
USING final AS s
ON t.occupancy_id = s.occupancy_id

WHEN MATCHED THEN UPDATE SET
    t.person_ref = s.person_ref,
    t.person_id = s.person_id,
    t.post_ref = s.post_ref,
    t.post_id = s.post_id,
    t.start_raw = s.start_raw,
    t.start_precision = s.start_precision,
    t.start_date = s.start_date,
    t.end_raw = s.end_raw,
    t.end_precision = s.end_precision,
    t.end_date = s.end_date,
    t.status_raw = s.status_raw,
    t.status_norm = s.status_norm,
    t.date_conflict = s.date_conflict,
    t.datasets = s.datasets,
    t.source_count = s.source_count,
    t._last_seen_snapshot = s._snapshot_version

WHEN NOT MATCHED THEN INSERT (
    occupancy_id,
    person_ref,
    person_id,
    post_ref,
    post_id,
    start_raw,
    start_precision,
    start_date,
    end_raw,
    end_precision,
    end_date,
    status_raw,
    status_norm,
    date_conflict,
    datasets,
    source_count,
    _first_seen_snapshot,
    _last_seen_snapshot
)
VALUES (
    s.occupancy_id,
    s.person_ref,
    s.person_id,
    s.post_ref,
    s.post_id,
    s.start_raw,
    s.start_precision,
    s.start_date,
    s.end_raw,
    s.end_precision,
    s.end_date,
    s.status_raw,
    s.status_norm,
    s.date_conflict,
    s.datasets,
    s.source_count,
    s._snapshot_version,
    s._snapshot_version
)

-- WHEN NOT MATCHED BY SOURCE THEN DELETE